# Computing distances between trajectories

Notebook to experiment with path similarity metrics. Currently testing Frechet distance through the similaritymeasures library. There is also a frechetdist library, but it appears that it may only allow 2D trajectories.

Notation for frechet_dist function is  frechet_dist(exp_data, num_data, p=2), where p controls the Minkowski p-norm (default: Euclidean, p=2).

In [53]:
# ~~~~ Imports ~~~~
import os, random, glob
import math
import pickle, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde, dirichlet
from scipy.signal import savgol_filter
from sklearn.isotonic import IsotonicRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from openTSNE import TSNE as oTSNE
from similaritymeasures import frechet_dist

# ~~~~ Environment setup ~~~~
device = torch.device('cpu')
print(f"Using device: {device}")

SEED = 29
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

from VAE_model_architectures import *

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

set_seed(SEED)

run_where = 'local'

# ~~~~ Configuration ~~~~
class Config:
    if run_where == 'great-lakes':
        BASE_DIR = "/nfs/turbo/umms-kaczoro/u19-shared/vae-HC-PFC/14mon_ADBXD_joined/"
        MODEL_PATH = os.path.join(BASE_DIR, "seed_model/C-GMVAE_SEED_38/saved_model_epoch20000.pth")
        GENE_CSV_PATH = "/nfs/turbo/umms-kaczoro/u19-shared/vae-HC-PFC/14mon_ADBXD_joined/input_datasets/hc_pfc_combined_rp_nor.csv"
        CFM_CSV_PATH  = "/nfs/turbo/umms-kaczoro/u19-shared/vae-HC-PFC/14mon_ADBXD_joined/input_datasets/CFC_CFM_residual_Bins_all_Strains.csv"
        LATENT_SPACE_CSV = os.path.join(BASE_DIR, "seed_model/C-GMVAE_SEED_38/latent_variables_epoch20000.csv")
        RECONS_DATA_CSV  = os.path.join(BASE_DIR, "seed_model/C-GMVAE_SEED_38/recons_cfm.csv")

    elif run_where == 'local':
        BASE_DIR = "/home/nghayes/latent-space-trajectories/"
        MODEL_PATH = os.path.join(BASE_DIR, "saved_model_epoch20000.pth")
        GENE_CSV_PATH = os.path.join(BASE_DIR, "hc_pfc_combined_rp_nor.csv")
        CFM_CSV_PATH  = os.path.join(BASE_DIR, "CFC_CFM_residual_Bins_all_Strains.csv")
        LATENT_SPACE_CSV = os.path.join(BASE_DIR, "latent_variables_epoch20000.csv")
        RECONS_DATA_CSV  = os.path.join(BASE_DIR, "recons_cfm.csv")

    GENE_COUNT = 4963
    D_in = D_out = GENE_COUNT
    LATENT_DIM = 10
    H1, H2, H3 = 128, 64, 32

    NUM_INTERPOLATION_STEPS = 24
    DENSITY_WEIGHTS = [0.1, 0.5, 0.9]
    DEFAULT_DENSITY_WEIGHT = 0.1
    NUM_PAIRS = 10
    KNN = 5
    

Using device: cpu


In [54]:
# loading a set of 10 trajectories using KDE mixture to test similarity metrics

with open('/home/nghayes/latent-space-trajectories/all_interpolations_mixturekdes_0.1_0.5_0.9.pkl', 'rb') as file:
    all_interpolations_mixture = pickle.load(file)

In [55]:
all_interpolations_mixture['pair_1']['paths'][0.5].shape

(31, 10)

In [56]:
curve1 = all_interpolations_mixture['pair_2']['paths'][0.5]
curve2 = all_interpolations_mixture['pair_4']['paths'][0.5]

for i in range(len(curve1)):
    print(np.linalg.norm(curve1[i]-curve2[i]))

3.1684246
3.9086483
3.7155898
3.5231893
3.331562
3.140848
2.9512255
2.7629187
2.5762153
2.3914926
2.2092466
2.0301445
1.855097
1.685369
1.5227385
1.3697373
1.2299654
1.1084379
1.0117494
0.94753534
0.9226007
0.9400773
0.99773914
1.0892198
1.2068567
1.3437971
1.4947447
1.6558721
1.8244843
2.9499512
2.1127195


In [57]:
frechet_dist(curve1, curve2, p=2)

np.float64(3.1684246215740743)

In [58]:
np.max(np.linalg.norm(curve1-curve2, axis=1)), np.argmax(np.linalg.norm(curve1-curve2, axis=1))

(np.float32(3.908648), np.int64(1))

In [59]:
curve1 = all_interpolations_mixture['pair_1']['paths'][0.5]

for i in range(len(all_interpolations_mixture) - 1):
    pair_string = 'pair_' + str(i+2)
    print(pair_string)
    curve2 = all_interpolations_mixture[pair_string]['paths'][0.5]
    print('distance between start points =', np.linalg.norm(curve1[0] - curve2[0]))
    print('distance between end points =', np.linalg.norm(curve1[-1] - curve2[-1]))
    print('distance between start + end for curve 1 =', np.linalg.norm(curve1[0]-curve1[-1]))
    print('distance between start + end for curve 2 =', np.linalg.norm(curve2[0]-curve2[-1]))
    print('farthest distance between two concurrent points =', np.max(np.linalg.norm(curve1-curve2, axis=1)))
    print('step with farthest distance =', np.argmax(np.linalg.norm(curve1-curve2, axis=1)))
    print('Frechet distance =', frechet_dist(curve1, curve2), '\n')

pair_2
distance between start points = 3.632018
distance between end points = 6.782375
distance between start + end for curve 1 = 113.36454
distance between start + end for curve 2 = 116.49176
farthest distance between two concurrent points = 6.792422
step with farthest distance = 29
Frechet distance = 6.782374746827199 

pair_3
distance between start points = 3.9894457
distance between end points = 7.127187
distance between start + end for curve 1 = 113.36454
distance between start + end for curve 2 = 116.41928
farthest distance between two concurrent points = 7.127187
step with farthest distance = 30
Frechet distance = 7.1271868682167385 

pair_4
distance between start points = 0.54699147
distance between end points = 8.876615
distance between start + end for curve 1 = 113.36454
distance between start + end for curve 2 = 121.558495
farthest distance between two concurrent points = 8.876615
step with farthest distance = 30
Frechet distance = 8.876614571737317 

pair_5
distance between